PART 2: AI integration (35%)
Requirements:
1. Design effective promps that include clinical context.
2. Handle API rate limits and errors gracefully.
3. Cache responses where appropriate to minimize API costs.
4. Document your prompt engineering approach.

**Requirement: Integrate Anthropic Claude**

In [42]:
# cell 1 - Installing the Claude library

!pip install anthropic

In [43]:
# cell 2 - Connecting to Claude
# Loading the Anthropic library into my notebook
import anthropic

# safely loading my API key from Colab Secrets
from google.colab import userdata
API_KEY = userdata.get('ANTHROPIC_API_KEY')

# Client = (saved as "client" can be used below)
# Client = our connection to Claude, using the secret key as proof it is me
client = anthropic.Anthropic(api_key=API_KEY)

print("Claude client is ready!")

Claude client is ready!


**Requirement: Cache responses where appropriate to minimize API costs**


In [44]:
#cell 3 - Caching Setup

cache = {} # empty dictionary to store our saved responses

def ask_claude(prompt):
    # check if we already have an answer for this exact prompt
    if prompt in cache:
        print('returning cached response - no API call needed!')
        return cache[prompt] # return saved answer for free


    # IF NOT IN CACHE, we call Claude normally
    response = client.messages.create(
        model='claude-sonnet-4-20250514',
        max_tokens=500, #500 words max for response
        messages=[
            {'role': 'user', 'content': prompt}
            #role "user" we are asking, content - message we send
        ]
)
    # save the response so we dont call Claude again
    # for the same question
    result = response.content[0].text
    cache[prompt] = result # store it in our cache dictionary

    return result

print('Cache is ready!')

Cache is ready!


**Requirement: Provide human-readable explanations**

In [45]:
# cell 4 - Sending our message to Claude
# testing that it can respond so we can understand

# ask_claude is used so the response gets cached
result = ask_claude('Say hello and confirm that we are ready!')
print(result)

Hello! 👋 

I'm ready and here to help! What would you like to work on together today?


**TEST** **DATA**

Sample patient records used to test both endpoints (examaples are from the assessment)

In [46]:
# cell 5 - Test Data
# patient_1 = used to test endpoint 1 (medication reconciliation)
# patient_2 = used to test endpoint 2 (data quality validation)

patient_1 = {
    'patient_context': {
        'age': 67,
        'conditions': ['Type 2 Diabetes', 'Hypertension'],
        'recent_labs': {'eGFR': 45} # kidney function score
    },
    'sources': [
        {
            'system': 'Hospital EHR',
            'medication': 'Metformin 1000mg twice daily',
            'last_updated': '2024-10-15',
            'source_reliability': 'high'
        },
        {
            'system': 'Primary Care',
            'medication': 'Metformin 500mg twice daily',
            'last_updated': '2025-01-20',
            'source_reliability': 'high'
        },
        {
            'system': 'Pharmacy',
            'medication': 'Metformin 1000mg daily',
            'last_filled': '2025-01-25',
            'source_reliability': 'medium'
        }
    ]
}

patient_2 = {
    # patient record with missing and implausible data
    'demographics': {'name': 'John Doe', 'dob': '1955-03-15', 'gender': 'M'},
    'medications': ['Metformin 500mg', 'Lisinopril 10mg'],
    'allergies': [], # empty
    'conditions': ['Type 2 Diabetes'],
    'vital_signs': {
        'blood_pressure': '340/180', # implausible - NEEDS TO BE FLAGGED!
        'heart_rate': 72
    },
    'last_updated': '2024-06-15' # old data
}

print('Test data ready!')


Test data ready!


**Requirement**: Generate clinical reasoning for reconciliation

**Requirement**: Design effective prompts with clinical context

In [47]:
# cell 6 - Sending REAL medication data to Claude

# core logic for endpoint 1: POST /api/reconcile/medication
     # (send medication data to this URL, get back a reconciled answer)
# ENDPOINT (URL that accepts my search and returns results)

# converting our patient_1 data (cell 5) into a readable message for Claude
medication_data = f'''
Patient age: {patient_1['patient_context']['age']}
Conditions: {', '.join(patient_1['patient_context']['conditions'])}
Lab results: eGFR {patient_1['patient_context']['recent_labs']['eGFR']} (kidney function)

Source 1 - {patient_1['sources'][0]['system']}: {patient_1['sources'][0]['medication']} (updated {patient_1['sources'][0]['last_updated']})
Source 2 - {patient_1['sources'][1]['system']}: {patient_1['sources'][1]['medication']} (updated {patient_1['sources'][1]['last_updated']})
Source 3 - {patient_1['sources'][2]['system']}: {patient_1['sources'][2]['medication']} (last filled {patient_1['sources'][2]['last_filled']})
'''

# using ask_claude() so response gets cached
result = ask_claude('Which medication record is most accurate? ' + medication_data)

# Claude explains its reasoning in plain English
print(result)


Based on the clinical information provided, **Source 2 (Primary Care record)** is most likely the most accurate.

Here's my reasoning:

**Clinical Considerations:**
- The patient has an eGFR of 45, indicating moderate kidney impairment (Stage 3b CKD)
- Standard guidelines recommend reducing metformin dose when eGFR is 30-45 mL/min/1.73m²
- The high dose in Source 1 (2000mg total daily) would be inappropriate for this level of kidney function

**Timeline Analysis:**
- Source 1: October 2024 (oldest, may not reflect dose adjustment after kidney function decline)
- Source 2: January 2025 (recent, likely reflects current clinical decision-making)
- Source 3: January 2025 (most recent fill, but could represent patient error or partial compliance)

**Most Likely Scenario:**
The primary care provider likely reduced the metformin dose from 1000mg twice daily to 500mg twice daily due to the patient's declining kidney function. The pharmacy record showing 1000mg daily might reflect:
- Patient co

**Requirement**: Generate clinical reasoning for reconciliation

**Requirement**: Design effective prompts with clinical context

**Requirement**: Provide human-readable explanations


In [48]:
# cell 7 - Asking Claude to respond in JSON format
# last thing to finish endpoint 1: POST /api/reconcile/medication

prompt_reconcile = f'''You are a clinical data reconciliation assistant.
Analyze these conflicting medication records and respond ONLY in JSON with these exact fields:
- reconciled_medication
- confidence_score (0.0 to 1.0)
- reasoning
- recommended_actions (a list)
- clinical_safety_check (PASSED or FLAGGED)

Patient age: {patient_1['patient_context']['age']}
Conditions: {', '.join(patient_1['patient_context']['conditions'])}
eGFR: {patient_1['patient_context']['recent_labs']['eGFR']}

Source 1 - {patient_1['sources'][0]['system']}: {patient_1['sources'][0]['medication']} (updated {patient_1['sources'][0]['last_updated']})
Source 2 - {patient_1['sources'][1]['system']}: {patient_1['sources'][1]['medication']} (updated {patient_1['sources'][1]['last_updated']})
Source 3 - {patient_1['sources'][2]['system']}: {patient_1['sources'][2]['medication']} (last filled {patient_1['sources'][2]['last_filled']})'''

# prompt engineering: we give Claude a role, clinical context,
# and tell it exactly how to format its response
# using ask_claude() so response gets cached
result = ask_claude(prompt_reconcile)
print(result)

```json
{
  "reconciled_medication": "Metformin 500mg twice daily",
  "confidence_score": 0.7,
  "reasoning": "Patient has eGFR of 45 mL/min/1.73m², indicating moderate kidney impairment (CKD stage 3b). Metformin dosing should be reduced when eGFR is 30-45. The primary care record shows the most recent clinical decision (2025-01-20) with appropriate dose reduction to 500mg twice daily, likely accounting for renal function. The pharmacy record (2025-01-25) shows 1000mg daily but may reflect dispensing of 500mg tablets twice daily.",
  "recommended_actions": [
    "Verify current renal function with recent eGFR/creatinine",
    "Confirm actual dispensed tablet strength and quantity with pharmacy",
    "Contact primary care provider to verify intended dosing given renal impairment",
    "Update hospital EHR to reflect current appropriate dosing",
    "Consider metformin discontinuation if eGFR continues to decline below 30"
  ],
  "clinical_safety_check": "FLAGGED"
}
```


**Requirement:** Detect implausible data

In [49]:
# cell 8 - Data Quality Validation
# This is endpoint 2: POST /api/validate/data-quality
# We send a patient_2 records to Claude and ask it to score the quality
# Claude will also detect impossible values (blood pressure 340/180)

# converting our patient_2 data (cell 5) into a readable message for Claude
patient_record = f'''
Demographics: {patient_2['demographics']['name']}, born {patient_2['demographics']['dob']}, {patient_2['demographics']['gender']}
Medications: {', '.join(patient_2['medications'])}
Allergies: {patient_2['allergies'] if patient_2['allergies'] else 'none listed'}
Conditions: {', '.join(patient_2['conditions'])}
Vital signs: blood pressure {patient_2['vital_signs']['blood_pressure']}, heart rate {patient_2['vital_signs']['heart_rate']}
Last updated: {patient_2['last_updated']}
'''

prompt_validate = f'''You are a clinical data quality auditor.
Review this patient record and respond ONLY in JSON with these exact fields:
- overall_score (0 to 100)
- breakdown with: completeness, accuracy, timeliness, clinical_plausibility (each 0 to 100)
- issues_detected (a list, each with: field, issue, severity as high/medium/low)

Patient record: {patient_record}'''

# prompt engineering: we give Claude a role, the full patient record,
# and tell it exactly what fields to score and what issues to look for
# using ask_claude() so response gets cached
result = ask_claude(prompt_validate)
print(result)

```json
{
  "overall_score": 25,
  "breakdown": {
    "completeness": 40,
    "accuracy": 10,
    "timeliness": 80,
    "clinical_plausibility": 10
  },
  "issues_detected": [
    {
      "field": "vital_signs_blood_pressure",
      "issue": "Blood pressure 340/180 is physiologically impossible and incompatible with life",
      "severity": "high"
    },
    {
      "field": "medications",
      "issue": "Missing dosing frequency for all medications",
      "severity": "medium"
    },
    {
      "field": "allergies",
      "issue": "Allergy status not properly documented - should specify 'NKDA' or 'No known allergies' rather than 'none listed'",
      "severity": "medium"
    },
    {
      "field": "conditions",
      "issue": "Missing ICD codes for documented conditions",
      "severity": "low"
    },
    {
      "field": "demographics",
      "issue": "Missing contact information and emergency contacts",
      "severity": "medium"
    },
    {
      "field": "clinical_correlation"

**Requirement**: Handle API rate limits and errors gracefully

In [50]:
# cell 9 - Error handling

# wrapping ask_claude() in try/except
# so if something goes wrong, we get a helpful message
# instead of a crash

try:
    result = ask_claude('confirm you are still working!')
    print(result)

except anthropic.RateLimitError:
    # RateLimitError = we sent too many requests too fast
    print('Slow down! Too many requests. Wait a moment and try again.')

except anthropic.APIError as e:
    # APIError = something else went wrong on Anthropic's side
    print('Something went wrong with the API: ' + str(e))

Yes, I'm working and ready to help! How can I assist you today?


**Requirement:** Cache responses where appropriate

Testing that cache works!

In [51]:
# cell 10 - Testing the cache
# run the same prompt twice
# second time should say 'returning cached response'
# proving we are saving money by not calling Claude twice

print('--- First call ---')
ask_claude('confirm you are still working!')

print('--- Second call (should be cached!) ---')
ask_claude('confirm you are still working!')

--- First call ---
--- Second call (should be cached!) ---
returning cached response - no API call needed!


"Yes, I'm working and ready to help! How can I assist you today?"